# Notebook for INFORMS 2019 Presentation

--- 
**IMPORTANT NOTE:**
- The examples below use 'OSRM-online' as the `dataProvider`.  This data provider has the advantage of not requiring an API key.  However, this is using the OSRM "demo" server.  This means that you're likely to encounter errors if the server is overloaded.
- If you get an error indicating that "OSRM is not available", I suggest that you switch to a different data provider.  See the following link for data provider options:
    - https://veroviz.org/docs/dataproviders.html
    - 'ORS-online' is the recommended data provider
    
---

Before continuing, please check out the installation instructions:
- https://veroviz.org/documentation.html

---

This notebook uses the following functions, each of which has extensive documentation (with examples):
- `generateNodes()` - https://veroviz.org/docs/veroviz.generateNodes.html#veroviz.generateNodes.generateNodes
- `createLeaflet()` - https://veroviz.org/docs/veroviz.createLeaflet.html#veroviz.createLeaflet.createLeaflet
- `addLeafletCircle()` - https://veroviz.org/docs/veroviz.createLeaflet.html#veroviz.createLeaflet.addLeafletCircle
- `addLeafletMarker()` - https://veroviz.org/docs/veroviz.createLeaflet.html#veroviz.createLeaflet.addLeafletMarker
- `getTimeDist2D()` - https://veroviz.org/docs/veroviz.getTimeDist2D.html#module-veroviz.getTimeDist2D
- `getTimeDist3D()` - https://veroviz.org/docs/veroviz.getTimeDist3D.html#module-veroviz.getTimeDist3D
- `createArcsFromNodeSeq()` - https://veroviz.org/docs/veroviz.generateArcs.html#veroviz.generateArcs.createArcsFromNodeSeq
- `getShapepoints2D()` - https://veroviz.org/docs/veroviz.getShapepoints2D.html#module-veroviz.getShapepoints2D
- `getShapepoints3D()` - https://veroviz.org/docs/veroviz.getShapepoints3D.html#module-veroviz.getShapepoints3D
- `addStaticAssignment()` - https://veroviz.org/docs/veroviz.buildAssignments.html#veroviz.buildAssignments.addStaticAssignment
- `createCesium()` - https://veroviz.org/docs/veroviz.createCesium.html#module-veroviz.createCesium

---

In [1]:
# Import the veroviz package, and make sure
# we're running the latest version:
import veroviz as vrv
vrv.checkVersion()

'Your current installed version of veroviz is 0.4.6, the latest version available is 0.4.5. To update to the latest version, type `pip install --upgrade veroviz` at a command-line prompt.'

## Generate test problems

Our first task will be to create a collection of 8 nodes:
- Node 0 will be the depot.  Its location should be generated from within a boundary region.  We'll color this node blue.
- Nodes 1-7 will be the customers.  They should be generated from a normal distribution, centered at the location of the INFORMS 2019 Annual Meeting, with a standard deviation of 1 mile.  These nodes will be red. 

In [2]:
# Define the region within which the depot node will be generated:
depotBoundary = [[47.66, -122.26], 
                 [47.52, -122.25], 
                 [47.50, -122.14], 
                 [47.71, -122.14]]

In [3]:
# Generate 1 depot node.
myNodes = vrv.generateNodes(numNodes        = 1,
                            startNode       = 0,
                            nodeType        = 'depot',
                            nodeName        = 'Depot',
                            leafletIconType = 'star',
                            nodeDistrib     = 'uniformBB',
                            nodeDistribArgs = {'boundingRegion': depotBoundary},
                            snapToRoad      = True,
                            dataProvider    = 'OSRM-online')

- See https://veroviz.org/docs/veroviz.generateNodes.html#veroviz.generateNodes.generateNodes for documentation on the `generateNodes()` function.

In [4]:
# We'll generate our customer nodes via a normal distribution.
# The center point will be the Washington State Convention Center (venue for INFORMS 2019).
# The standard deviation should be expressed in units of meters.
WashStConvCtr = [47.6114, -122.3323]
radiusMeters  = vrv.convertDistance(2, 'mi', 'meters')

In [5]:
# Generate 7 customers nodes.  
# We'll assign these to the category of 'customer', so we can filter 
# for these nodes later if we wish.  Each node will be uniquely named 
# as 'Cust x', where `x` will be replaced by the numbers 1 thru 7.
myNodes = vrv.generateNodes(initNodes       = myNodes,
                            numNodes        = 7,
                            startNode       = 1,
                            nodeType        = 'customer',
                            nodeName        = 'Cust ',
                            incrementName   = True, 
                            leafletColor    = 'red',
                            leafletIconType = 'home',
                            cesiumColor     = 'Cesium.Color.RED',
                            nodeDistrib     = 'normal',
                            nodeDistribArgs = {'center': WashStConvCtr,
                                               'stdDev': radiusMeters},
                            snapToRoad      = True,
                            dataProvider    = 'OSRM-online')

In [6]:
# Display our nodes dataframe, which now contains the depot and 7 customers.
myNodes

,id,lat,lon,altMeters,nodeName,nodeType,popupText,leafletIconPrefix,leafletIconType,leafletColor,leafletIconText,cesiumIconType,cesiumColor,cesiumIconText,elevMeters
0,0,47.567258,-122.189889,0.0,Depot,depot,0,glyphicon,star,blue,0,pin,blue,0,None
1,1,47.608239,-122.324083,0.0,Cust 1,customer,1,glyphicon,home,red,1,pin,red,1,None
2,2,47.599200,-122.327818,0.0,Cust 2,customer,2,glyphicon,home,red,2,pin,red,2,None
3,3,47.627743,-122.311275,0.0,Cust 3,customer,3,glyphicon,home,red,3,pin,red,3,None
4,4,47.617816,-122.358286,0.0,Cust 4,customer,4,glyphicon,home,red,4,pin,red,4,None
5,5,47.621129,-122.362482,0.0,Cust 5,customer,5,glyphicon,home,red,5,pin,red,5,None
6,6,47.623110,-122.357715,0.0,Cust 6,customer,6,glyphicon,home,red,6,pin,red,6,None
7,7,47.617081,-122.357265,0.0,Cust 7,customer,7,glyphicon,home,red,7,pin,red,7,None


- See https://veroviz.org/docs/nodes.html for documentation of the 'nodes' dataframe structure.

## Visualize Nodes on a Map
The following cells describe how to display information about the distributions from which our nodes were drawn.  We'll also display the nodes themselves on the map.

In [7]:
# Let's begin by visualizing the distribution parameters.

# Start with an empty map and draw the depot boundary polygon:
myMap = vrv.createLeaflet(boundingRegion = depotBoundary)

# Now, draw some circles to represent the center and spread of the normal 
# distribution from which our customer locations are drawn.
# Each circle will be centered at the Convention Center. 
# We'll show circles of 1, 2, and 3 standard deviations.
myMap = vrv.addLeafletCircle(mapObject = myMap,          center    = WashStConvCtr,
                             radius    = 3*radiusMeters, fillColor = 'red')
myMap = vrv.addLeafletCircle(mapObject = myMap,          center    = WashStConvCtr,
                             radius    = 2*radiusMeters, fillColor = 'green')
myMap = vrv.addLeafletCircle(mapObject = myMap,          center    = WashStConvCtr,
                             radius    = 1*radiusMeters, fillColor = 'blue')

# Use a circle marker to denote the center point:
myMap = vrv.addLeafletMarker(mapObject = myMap,          center    = WashStConvCtr)


# Add some text labels to help explain the map:
myMap = vrv.addLeafletText(mapObject   = myMap, 
                           anchorPoint = WashStConvCtr, 
                           text        = "INFORMS", 
                           fontSize    = 12, fontColor = 'white')
myMap = vrv.addLeafletText(mapObject   = myMap, 
                           anchorPoint = [47.6598, -122.148], 
                           text        = "1 Depot",
                           fontSize    = 12, fontColor = 'darkred',
                           horizAlign  = 'right')
myMap = vrv.addLeafletText(mapObject   = myMap, 
                           anchorPoint = vrv.pointInDistance2D(WashStConvCtr, 180, radiusMeters), 
                           text        = "1 std. dev.",
                           fontSize    = 12, fontColor = 'black')
myMap = vrv.addLeafletText(mapObject   = myMap, 
                           anchorPoint = vrv.pointInDistance2D(WashStConvCtr, 180, 3*radiusMeters), 
                           text        = "3 std. devs.",
                           fontSize    = 12, fontColor = 'black')
myMap = vrv.addLeafletText(mapObject   = myMap, 
                           anchorPoint = vrv.pointInDistance2D(WashStConvCtr, 0, 2.5*radiusMeters), 
                           text        = "7 Customers",
                           fontSize    = 12, fontColor = 'darkred')
myMap

In [8]:
# Now let's add our nodes to the map:
vrv.createLeaflet(mapObject     = myMap,
                  mapBackground = 'cartodb positron', 
                  nodes         = myNodes)

- We can produce maps of just the nodes (without all of the distribution graphics).
- The following examples also show some of the available map backgrounds.  See https://veroviz.org/docs/leaflet_style.html#leaflet-map-background-map-tiles for a complete list of options.

In [9]:
vrv.createLeaflet(mapBackground = 'cartodb dark_matter', 
                  nodes         = myNodes)

In [10]:
vrv.createLeaflet(mapBackground = 'Arcgis Aerial', 
                  nodes         = myNodes)

In [11]:
vrv.createLeaflet(mapBackground = 'Open Topo', 
                  nodes         = myNodes)

In [12]:
vrv.createLeaflet(mapBackground = 'Arcgis Ocean', 
                  nodes         = myNodes)

Detailed documentation of the Leaflet-related functions may be found here:
- `createLeaflet()` - https://veroviz.org/docs/veroviz.createLeaflet.html#veroviz.createLeaflet.createLeaflet
- `addLeafletCircle()` - https://veroviz.org/docs/veroviz.createLeaflet.html#veroviz.createLeaflet.addLeafletCircle
- `addLeafletMarker()` - https://veroviz.org/docs/veroviz.createLeaflet.html#veroviz.createLeaflet.addLeafletMarker

## Travel Matrices

Now that we've created some nodes, our next task is to find the time and distance to travel between each pair of nodes.

We'll want to do this for two different types of vehicles:
1. Trucks, which should travel along a road network and obey speed limits, and 
2. Drones, which won't follow the road, but will have a vertical component to their travel.

In [13]:
# Find the truck's travel time (in seconds) and distance (in meters) 
# between all pairs of nodes.
# We're looking for the fastest time.  Alternatively, we could look
# for the shortest distance.
# We'll need a data provider for this information.
[timeSec, distMeters] = vrv.getTimeDist2D(nodes        = myNodes, 
                                          routeType    = 'fastest',
                                          dataProvider = 'OSRM-online')

In [14]:
# Let's find the travel time from node 1 to node 3:
timeSec[1, 3]

368.8

In [15]:
# Similarly, we can find the distance from node 1 to node 3:
distMeters[1,3]

2892.1

In [16]:
# `timeSec` is a dictionary.  
# We could print it, but it's difficult to read:
timeSec

{(0, 0): 0.0,
 (0, 1): 1082.5,
 (0, 2): 1061.0,
 (0, 3): 1386.2,
 (0, 4): 1440.4,
 (0, 5): 1476.2,
 (0, 6): 1443.2,
 (0, 7): 1413.5,
 (1, 0): 1085.6,
 (1, 1): 0.0,
 (1, 2): 209.2,
 (1, 3): 368.8,
 (1, 4): 481.0,
 (1, 5): 492.2,
 (1, 6): 477.2,
 (1, 7): 454.1,
 (2, 0): 1046.0,
 (2, 1): 199.5,
 (2, 2): 0.0,
 (2, 3): 506.7,
 (2, 4): 468.9,
 (2, 5): 510.3,
 (2, 6): 495.3,
 (2, 7): 442.0,
 (3, 0): 1371.4,
 (3, 1): 373.4,
 (3, 2): 516.8,
 (3, 3): 0.0,
 (3, 4): 648.9,
 (3, 5): 632.8,
 (3, 6): 617.8,
 (3, 7): 622.0,
 (4, 0): 1353.6,
 (4, 1): 506.0,
 (4, 2): 430.4,
 (4, 3): 622.6,
 (4, 4): 0.0,
 (4, 5): 142.0,
 (4, 6): 170.6,
 (4, 7): 26.9,
 (5, 0): 1400.0,
 (5, 1): 526.3,
 (5, 2): 481.9,
 (5, 3): 633.0,
 (5, 4): 108.8,
 (5, 5): 0.0,
 (5, 6): 87.3,
 (5, 7): 103.8,
 (6, 0): 1388.1,
 (6, 1): 520.4,
 (6, 2): 495.0,
 (6, 3): 590.7,
 (6, 4): 160.3,
 (6, 5): 91.6,
 (6, 6): 0.0,
 (6, 7): 155.3,
 (7, 0): 1326.7,
 (7, 1): 479.1,
 (7, 2): 403.5,
 (7, 3): 595.7,
 (7, 4): 26.9,
 (7, 5): 115.1,
 (7, 6): 143

In [17]:
# Instead, we can convert this dictionary to a dataframe 
# so we can view the travel times in matrix form:
vrv.convertMatricesDictionaryToDataframe(timeSec)

,0,1,2,3,4,5,6,7
0,0.0,1082.5,1061.0,1386.2,1440.4,1476.2,1443.2,1413.5
1,1085.6,0.0,209.2,368.8,481.0,492.2,477.2,454.1
2,1046.0,199.5,0.0,506.7,468.9,510.3,495.3,442.0
3,1371.4,373.4,516.8,0.0,648.9,632.8,617.8,622.0
4,1353.6,506.0,430.4,622.6,0.0,142.0,170.6,26.9
5,1400.0,526.3,481.9,633.0,108.8,0.0,87.3,103.8
6,1388.1,520.4,495.0,590.7,160.3,91.6,0.0,155.3
7,1326.7,479.1,403.5,595.7,26.9,115.1,143.7,0.0


In [18]:
# Similarly, we can look at a matrix of travel distances:
vrv.convertMatricesDictionaryToDataframe(distMeters)

,0,1,2,3,4,5,6,7
0,0.0,16586.9,16415.4,19713.7,19389.4,19816.5,20768.8,19277.2
1,16423.6,0.0,1519.5,2892.1,3591.4,3702.4,3691.8,3479.2
2,16177.3,1478.5,0.0,4576.4,3504.3,3937.8,3927.2,3392.1
3,19709.5,2847.0,4057.0,0.0,4919.5,4945.4,4934.8,4807.3
4,18847.7,3775.4,3328.1,4733.2,0.0,888.4,1159.1,112.2
5,19857.3,3934.8,3862.0,4945.1,648.2,0.0,564.0,768.0
6,20400.5,3924.2,3817.2,4608.3,1062.0,564.0,0.0,1181.7
7,18735.5,3663.2,3215.9,4621.0,112.2,776.3,1047.0,0.0


For the drones, we'll need to calculate the travel matrices differently, as they travel in 3 dimensions.  In particular, we'll make the following assumptions:
- The drones will takeoff and land vertically, and 
- They will travel horizontally at a constant altitude

In [19]:
# The `getTimeDist3D` function returns 3 dictionaries.
# - timeSec3D will contain the travel times (in seconds) between each pair of nodes.
# - groundDist will describe only the ground distance between each pair of nodes 
#   (i.e., without any vertical travel).
# - distMeters3D will contain the total distance traveled between each pair of nodes
#   (i.e., both the horizontal and vertical travel).
[timeSec3D, groundDist, distMeters3D] = vrv.getTimeDist3D(
    nodes              = myNodes, 
    routeType          = 'square',
    takeoffSpeedMPS    = vrv.convertSpeed(20, 'mi', 'hr', 'm', 's'),
    landSpeedMPS       = vrv.convertSpeed(5, 'mi', 'hr', 'm', 's'),
    cruiseAltMetersAGL = 100, 
    cruiseSpeedMPS     = vrv.convertSpeed(50, 'mi', 'hr', 'm', 's'))

In [20]:
vrv.convertMatricesDictionaryToDataframe(distMeters3D)

,0,1,2,3,4,5,6,7
0,0.000000,11274.525233,11166.582711,11538.306085,14056.682664,14495.711366,14266.602035,13953.493489
1,11274.525233,0.000000,1243.485846,2572.655425,2983.181361,3422.978369,3221.002542,2881.409416
2,11166.582711,1243.485846,0.000000,3608.501659,3287.377060,3768.886680,3681.311932,3175.656830
3,11538.306085,2572.655425,3608.501659,0.000000,3902.065998,4118.610107,3728.416284,3854.588861
4,14056.682664,2983.181361,3287.377060,3902.065998,0.000000,684.946499,790.165410,312.112661
5,14495.711366,3422.978369,3768.886680,4118.610107,684.946499,0.000000,620.608296,796.962996
6,14266.602035,3221.002542,3681.311932,3728.416284,790.165410,620.608296,0.000000,871.174861
7,13953.493489,2881.409416,3175.656830,3854.588861,312.112661,796.962996,871.174861,0.000000


In [21]:
# Here's the total distance to travel from node 1 to node 3:
distMeters3D[1, 3]

# This includes 100 meters for takeoff and 100 meters to land.

np.float64(2572.655425386344)

In [22]:
# The grond distance ignores the change in altitude:
groundDist[1, 3]

np.float64(2372.655425386344)

## Creating a Solution (this would be your solver)
VeRoViz isn't a solver...it's a toolkit for visualizing solutions generated by **your** solver.

In this section we provide some sample code to mimic a solver.  

In [23]:
# We'll need pandas to concatenate dataframes
import pandas as pd

We're going to define a function that will produce routes for our truck and drones.

There's no algorithm here...just a hard-coded assignment of routes. 
The important thing in this function is that it returns some information which 
we'll convert for use by VeRoViz.

Although the "solver" logic is lousy, hopefully it will help you understand one approach for converting your solver's output into a form usable by VeRoViz.

In [29]:
def BLACK_BOX_SOLVER(myNodes, timeSec, timeSec3D):
    # THIS IS A VERY NAIVE "SOLUTION" TO THE PROBLEM OF 
    # VISITING CUSTOMERS WITH BOTH A TRUCK AND MULTIPLE DRONES.
    #
    # THIS FUNCTION IS PROVIDED SIMPLY FOR DEMONSTRATION 
    # PURPOSES. 

    # ------------------------------------------------------
    # Assumptions:
    # - The depot is node 0.  
    # - The truck originates/terminates at the depot.
    # - The truck has a fixed route (defined below).
    # - The drones also have fixed routes (also below).
    # - This "solver" requires exactly 1 depot (node 0) 
    #   and 7 customers (nodes 1-7).
    # - Customer deliveries require 30 seconds (for both 
    #   the truck and drones).
    # - Drones must wait for the truck to land.
    # - Drone launches and recoveries happen instantaneously.
    # - The truck can launch/recovery multiple UAVs 
    #   simultaneously.
    # - Drones have unlimited endurance, and can delivery
    #   any of the packages.
    # ------------------------------------------------------

    # ------------------------------------------------------
    # Here are hard-coded routes for the truck and 2 drones:
    # ------------------------------------------------------
    truckRoute    = [0, 1, 3, 5, 0]
    droneSorties = {
        1: [[1, 2, 3], [3, 4, 5]],
        2: [[1, 6, 5], [5, 7, 0]]
    }

    serviceTime = 30  # seconds
    
    styles = {'truck': {'color': 'blue', 'line': 'solid'},
              'drone1': {'color': 'red', 'line': 'dashed'},
              'drone2': {'color': 'purple', 'line': 'dotted'}
             }
    
    # ------------------------------------------------------
    # Generate an "arcs" dataframe to visualize 
    # the sequence of visits (without any timing details).
    # ------------------------------------------------------
    # Truck - blue route
    myArcs = vrv.createArcsFromNodeSeq(nodeSeq = truckRoute,
                                       nodes   = myNodes, 
                                       leafletColor = styles['truck']['color'],
                                       leafletStyle = styles['truck']['line'])
    # Drone 1 - red dashed routes
    for sortie in droneSorties[1]:
        myArcs = vrv.createArcsFromNodeSeq(initArcs     = myArcs,
                                           nodeSeq      = sortie,
                                           nodes        = myNodes,
                                           leafletColor = styles['drone1']['color'],
                                           leafletStyle = styles['drone1']['line'], 
                                           useArrows    = False)
    # Drone 2 - purple dotted routes
    for sortie in droneSorties[2]:
        myArcs = vrv.createArcsFromNodeSeq(initArcs     = myArcs,
                                           nodeSeq      = sortie,
                                           nodes        = myNodes,
                                           leafletColor = styles['drone2']['color'],
                                           leafletStyle = styles['drone2']['line'], 
                                           useArrows    = False)


    # ------------------------------------------------------
    # Generate a detailed solution, including the timing
    # of visits for each vehicle.
    # ------------------------------------------------------
    solution = {}

    solution['truck'] = []
    solution['drone1'] = []
    solution['drone2'] = []

    # We'll need to figure out when the truck can leave each node
    # (after waiting for drones).
    maxDepartureTime = {}
    for i in truckRoute:
        maxDepartureTime[i] = 0.0

    # Start building the detailed solution, with timing:
    truckStartTime = 0.0    
    for index in list(range(0, len(truckRoute)-1)):
        truckStartNode = truckRoute[index]
        truckEndNode = truckRoute[index+1]
        truckArrivalTime = truckStartTime + timeSec[truckStartNode, truckEndNode]
        truckEndServiceTime = truckArrivalTime + serviceTime

        maxDepartureTime[truckEndNode] = max(maxDepartureTime[truckEndNode], 
                                             truckEndServiceTime)

        for drone in droneSorties:
            for sortie in droneSorties[drone]:
                if (sortie[0] == truckStartNode):

                    # i to j
                    droneStartTime = truckStartTime
                    droneArrivalTime = droneStartTime + timeSec3D[sortie[0], sortie[1]]
                    droneEndServiceTime = droneArrivalTime + serviceTime
                    solution['drone%d' % (drone)].append({
                        'startNode': sortie[0],
                        'startTime': droneStartTime,
                        'status': 'delivery',
                        'modelFile': 'veroviz/models/drone_package.glb',
                        'arcColor': styles['drone%d' % (drone)]['color'],
                        'arcStyle': styles['drone%d' % (drone)]['line'],
                        'endNode': sortie[1],
                        'droneArrivalTime': droneArrivalTime,
                        'truckArrivalTime': -1,
                        'serviceStartTime': droneArrivalTime,
                        'serviceEndTime': droneEndServiceTime,
                        'endTime': droneEndServiceTime,
                        'packageFile': 'veroviz/models/box_yellow.gltf'
                    })

                    # j to k
                    droneStartTime = droneEndServiceTime
                    droneArrivalTime = droneStartTime + timeSec3D[sortie[1], sortie[2]]
                    droneEndServiceTime = droneArrivalTime
                    solution['drone%d' % (drone)].append({
                        'startNode': sortie[1],
                        'startTime': droneStartTime,

                        'status': 'empty',
                        'modelFile': 'veroviz/models/drone.glb',
                        'arcColor': styles['drone%d' % (drone)]['color'],
                        'arcStyle': styles['drone%d' % (drone)]['line'],
                        'endNode': sortie[2],
                        'droneArrivalTime': droneArrivalTime,
                        'truckArrivalTime': -1,   # still unknown
                        'serviceStartTime': droneArrivalTime,
                        'serviceEndTime': droneEndServiceTime,
                        'endTime': droneEndServiceTime,
                        'packageFile': None
                    })

                    maxDepartureTime[sortie[2]] = max(maxDepartureTime[sortie[2]], 
                                                    droneEndServiceTime)

        solution['truck'].append({
            'startNode': truckStartNode,
            'startTime': truckStartTime,
            'status': 'delivery' if truckEndNode != 0 else 'empty',
            'modelFile': 'veroviz/models/ub_truck.glb',
            'arcColor': styles['truck']['color'],
            'arcStyle': styles['truck']['line'],
            'endNode': truckEndNode,
            'droneArrivalTime': -1, # ignored
            'truckArrivalTime': truckArrivalTime,
            'serviceStartTime': truckArrivalTime,
            'serviceEndTime': truckEndServiceTime,
            'endTime': maxDepartureTime[truckEndNode],
            'packageFile': 'veroviz/models/box_blue.gltf' if truckEndNode != 0 else None
        })

        truckStartTime = maxDepartureTime[truckEndNode]

    # Now, update the solution for each drone.
    # When will the truck be there to pick it up?
    for drone in droneSorties:
        for index in list(range(0, len(solution['drone%d' % (drone)]))):
            droneEndNode = solution['drone%d' % (drone)][index]['endNode']
            if (solution['drone%d' % (drone)][index]['status'] == 'empty'):
                solution['drone%d' % (drone)][index]['truckArrivalTime'] = maxDepartureTime[droneEndNode]

    return (myArcs, solution)

In [30]:
# Let's run our dummy solver:
[myArcs, mySolution] = BLACK_BOX_SOLVER(myNodes, timeSec, timeSec3D)

# The warnings here are to be expected.  
# VeRoViz will automatically assign unique odIDs (origin/destination) for each arc.

## Converting Solution into VeRoViz Dataframes

We have now obtained two outputs from our dummy solver:
1. `myArcs` describes just the sequence of node visits for each vehicle.
    - It does not contain any information about the timing of each visit, nor does it describe the turn-by-turn movement of vehicles along these arcs.
    - However, this information can still be useful to get an overall sense of the routes.
2. `mySolution` provides detailed information about the vehicle movements. 
    - This includes the time that vehicles start from, and end at, each origin/destination pair.
    - It also includes any time that a drone waits for a truck (or vice versa).
    - Additionally, each origin/destination pair is decomposed into smaller segments.  For the truck, these segments represent turn-by-turn travel along the road network.  For the drones, this includes takeoff and landing.

In [31]:
# Display our arcs dataframe.
myArcs

,odID,objectID,startLat,startLon,endLat,endLon,leafletColor,leafletWeight,leafletStyle,leafletOpacity,leafletCurveType,leafletCurvature,useArrows,cesiumColor,cesiumWeight,cesiumStyle,cesiumOpacity,popupText,startElevMeters,endElevMeters
0,1,None,47.567258,-122.189889,47.608239,-122.324083,blue,3,solid,0.8,straight,0,True,orange,3,solid,0.8,None,None,None
1,2,None,47.608239,-122.324083,47.627743,-122.311275,blue,3,solid,0.8,straight,0,True,orange,3,solid,0.8,None,None,None
2,3,None,47.627743,-122.311275,47.621129,-122.362482,blue,3,solid,0.8,straight,0,True,orange,3,solid,0.8,None,None,None
3,4,None,47.621129,-122.362482,47.567258,-122.189889,blue,3,solid,0.8,straight,0,True,orange,3,solid,0.8,None,None,None
4,5,None,47.608239,-122.324083,47.599200,-122.327818,red,3,dashed,0.8,straight,0,False,orange,3,solid,0.8,None,None,None
5,6,None,47.599200,-122.327818,47.627743,-122.311275,red,3,dashed,0.8,straight,0,False,orange,3,solid,0.8,None,None,None
6,7,None,47.627743,-122.311275,47.617816,-122.358286,red,3,dashed,0.8,straight,0,False,orange,3,solid,0.8,None,None,None
7,8,None,47.617816,-122.358286,47.621129,-122.362482,red,3,dashed,0.8,straight,0,False,orange,3,solid,0.8,None,None,None
8,9,None,47.608239,-122.324083,47.623110,-122.357715,purple,3,dotted,0.8,straight,0,False,orange,3,solid,0.8,None,None,None
9,10,None,47.623110,-122.357715,47.621129,-122.362482,purple,3,dotted,0.8,straight,0,False,orange,3,solid,0.8,None,None,None


- Note:  "arcs" dataframes have a specific pre-defined structure within VeRoViz.  See https://veroviz.org/docs/arcs.html for details.

In [32]:
# Take a look at the straight-line arcs and nodes:
vrv.createLeaflet(arcs = myArcs, nodes = myNodes)

In [33]:
# `mySolution` is a dictionary.  
# We can look at this dictionary, but it's not easily readable:
mySolution

{'truck': [{'startNode': 0,
   'startTime': 0.0,
   'status': 'delivery',
   'modelFile': 'veroviz/models/ub_truck.glb',
   'arcColor': 'blue',
   'arcStyle': 'solid',
   'endNode': 1,
   'droneArrivalTime': -1,
   'truckArrivalTime': 1082.5,
   'serviceStartTime': 1082.5,
   'serviceEndTime': 1112.5,
   'endTime': 1112.5,
   'packageFile': 'veroviz/models/box_blue.gltf'},
  {'startNode': 1,
   'startTime': 1112.5,
   'status': 'delivery',
   'modelFile': 'veroviz/models/ub_truck.glb',
   'arcColor': 'blue',
   'arcStyle': 'solid',
   'endNode': 3,
   'droneArrivalTime': -1,
   'truckArrivalTime': 1481.3,
   'serviceStartTime': 1481.3,
   'serviceEndTime': 1511.3,
   'endTime': 1511.3,
   'packageFile': 'veroviz/models/box_blue.gltf'},
  {'startNode': 3,
   'startTime': 1511.3,
   'status': 'delivery',
   'modelFile': 'veroviz/models/ub_truck.glb',
   'arcColor': 'blue',
   'arcStyle': 'solid',
   'endNode': 5,
   'droneArrivalTime': -1,
   'truckArrivalTime': 2144.1,
   'serviceStartT

In [34]:
# Instead, it might be easier to view the solution as a dataframe, 
# looking at one vehicle at a time:
pd.DataFrame(mySolution['truck'])

,startNode,startTime,status,modelFile,arcColor,arcStyle,endNode,droneArrivalTime,truckArrivalTime,serviceStartTime,serviceEndTime,endTime,packageFile
0,0,0.0,delivery,veroviz/models/ub_truck.glb,blue,solid,1,-1,1082.5,1082.5,1112.5,1112.5,veroviz/models/box_blue.gltf
1,1,1112.5,delivery,veroviz/models/ub_truck.glb,blue,solid,3,-1,1481.3,1481.3,1511.3,1511.3,veroviz/models/box_blue.gltf
2,3,1511.3,delivery,veroviz/models/ub_truck.glb,blue,solid,5,-1,2144.1,2144.1,2174.1,2174.1,veroviz/models/box_blue.gltf
3,5,2174.1,empty,veroviz/models/ub_truck.glb,blue,solid,0,-1,3574.1,3574.1,3604.1,3604.1,None


In [35]:
pd.DataFrame(mySolution['drone1'])

,startNode,startTime,status,modelFile,arcColor,arcStyle,endNode,droneArrivalTime,truckArrivalTime,serviceStartTime,serviceEndTime,endTime,packageFile
0,1,1112.500000,delivery,veroviz/models/drone_package.glb,red,dashed,2,1215.107890,-1.0,1215.107890,1245.107890,1245.107890,veroviz/models/box_yellow.gltf
1,2,1245.107890,empty,veroviz/models/drone.glb,red,dashed,3,1453.523836,1511.3,1453.523836,1453.523836,1453.523836,None
2,3,1511.300000,delivery,veroviz/models/drone_package.glb,red,dashed,4,1732.849674,-1.0,1732.849674,1762.849674,1762.849674,veroviz/models/box_yellow.gltf
3,4,1762.849674,empty,veroviz/models/drone.glb,red,dashed,5,1840.469162,2174.1,1840.469162,1840.469162,1840.469162,None


- NOTE:  The structure of `mySolution` isn't specific to VeRoViz.  Your solver does **not** need to produce an output that looks exactly like this.  
    - In the code below, we'll convert this information into a VeRoViz "assignments" dataframe, which **does** have a clearly-defined structure.
    - See https://veroviz.org/docs/assignments.html for details on "assignments" dataframes.

In [36]:
# Initialize an empty "assignments" dataframe.  
assignmentsDF = vrv.initDataframe('assignments')

for v in mySolution:
    for leg in mySolution[v]:
        # Vehicle moves between two different locations:
        if (v == 'truck'):
            # Movement - 2D
            myShapepoints = vrv.getShapepoints2D(
                objectID       = v,
                modelFile      = leg['modelFile'],
                startLoc       = list(myNodes[myNodes['id'] == leg['startNode']][['lat', 'lon']].values[0]),
                endLoc         = list(myNodes[myNodes['id'] == leg['endNode']][['lat', 'lon']].values[0]),
                startTimeSec   = leg['startTime'],
                expDurationSec = leg['truckArrivalTime'] - leg['startTime'],
                routeType      = 'fastest',
                dataProvider   = 'OSRM-online',
                leafletColor   = leg['arcColor'],
                leafletStyle   = leg['arcStyle'],
                cesiumColor    = 'Cesium.Color.%s' % (leg['arcColor'].upper()),
                cesiumStyle    = leg['arcStyle'])            
        else:
            # movement - 3D            
            myShapepoints = vrv.getShapepoints3D(
                objectID           = v,
                modelFile          = leg['modelFile'],
                startLoc           = list(myNodes[myNodes['id'] == leg['startNode']][['lat', 'lon']].values[0]),
                endLoc             = list(myNodes[myNodes['id'] == leg['endNode']][['lat', 'lon']].values[0]),
                startTimeSec       = leg['startTime'],
                routeType          = 'square',
                takeoffSpeedMPS    = vrv.convertSpeed(20, 'mi', 'hr', 'm', 's'),
                landSpeedMPS       = vrv.convertSpeed(5, 'mi', 'hr', 'm', 's'),
                cruiseAltMetersAGL = 100, 
                cruiseSpeedMPS     = vrv.convertSpeed(50, 'mi', 'hr', 'm', 's'),
                earliestLandTime   = leg['truckArrivalTime'],
                loiterPosition     = 'arrivalAtAlt',
                leafletColor       = leg['arcColor'],
                leafletStyle       = leg['arcStyle'],
                cesiumColor        = 'Cesium.Color.%s' % (leg['arcColor'].upper()),
                cesiumStyle        = leg['arcStyle']) 

        assignmentsDF = pd.concat([assignmentsDF, myShapepoints], 
                                  ignore_index=True, sort=False)        

        # Vehicle is static (service or waiting for drone):
        if (leg['endTime'] > max(leg['droneArrivalTime'], leg['truckArrivalTime'])):
            # static
            assignmentsDF = vrv.addStaticAssignment(
                initAssignments = assignmentsDF, 
                objectID        = v,
                modelFile       = leg['modelFile'],
                loc             = list(myNodes[myNodes['id'] == leg['endNode']][['lat', 'lon']].values[0]),
                startTimeSec    = max(leg['droneArrivalTime'], leg['truckArrivalTime']),
                endTimeSec      = leg['endTime'])

        # Leave a package at the customer:
        if (leg['status'] == 'delivery'):
            assignmentsDF = vrv.addStaticAssignment(
                initAssignments = assignmentsDF, 
                objectID        = 'package %d' % leg['endNode'],
                modelFile       = leg['packageFile'], 
                modelMinPxSize  = 45,
                loc             = list(myNodes[myNodes['id'] == leg['endNode']][['lat', 'lon']].values[0]),
                startTimeSec    = leg['endTime'],
                endTimeSec      = -1)   
            

/tmp/ipykernel_99504/309690072.py:42: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  assignmentsDF = pd.concat([assignmentsDF, myShapepoints],


In [37]:
# Here's our VeRoViz "assignments" dataframe.
# Note that it has many more rows (and a few more columns) than the "arcs" dataframe.
assignmentsDF

,odID,objectID,modelFile,modelScale,modelMinPxSize,startTimeSec,startLat,startLon,startAltMeters,endTimeSec,...,ganttColor,popupText,startElevMeters,endElevMeters,wayname,waycategory,surface,waytype,steepness,tollway
0,1,truck,/veroviz/models/ub_truck.glb,100,75,0,47.567258,-122.189889,0.0,6,...,darkgray,None,0.0,0.0,None,None,None,None,0,False
1,1,truck,/veroviz/models/ub_truck.glb,100,75,6,47.567240,-122.188704,0.0,15,...,darkgray,None,0.0,0.0,None,None,None,None,0,False
2,1,truck,/veroviz/models/ub_truck.glb,100,75,15,47.567218,-122.186894,0.0,27,...,darkgray,None,0.0,0.0,None,None,None,None,0,False
3,1,truck,/veroviz/models/ub_truck.glb,100,75,27,47.567178,-122.184531,0.0,52,...,darkgray,None,0.0,0.0,None,None,None,None,0,False
4,1,truck,/veroviz/models/ub_truck.glb,100,75,52,47.570023,-122.181921,0.0,58,...,darkgray,None,0.0,0.0,None,None,None,None,0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
254,16,package 7,/veroviz/models/box_yellow.gltf,100,45,2286,47.617081,-122.357265,0.0,-1,...,darkgray,None,0.0,0.0,None,None,None,None,0,False
255,1,drone2,/veroviz/models/drone.glb,100,75,2286,47.617081,-122.357265,0.0,2297,...,darkgray,None,0.0,0.0,None,None,None,None,0,False
256,1,drone2,/veroviz/models/drone.glb,100,75,2297,47.617081,-122.357265,100.0,2913,...,darkgray,None,0.0,0.0,None,None,None,None,0,False
257,1,drone2,/veroviz/models/drone.glb,100,75,2913,47.567258,-122.189889,100.0,3559,...,lightgray,None,0.0,0.0,None,None,None,None,0,False


In [38]:
# We can visualize the 2D components of the assignments
# in a Leaflet map.  Note that the truck route along the 
# road is now displayed.  The drone routes don't look 
# much different from the previous "myArcs" dataframe, 
# as Leaflet is unable to represent the vertical components 
# of the drones' flight.
vrv.createLeaflet(arcs = assignmentsDF, nodes = myNodes, useArrows = False)

## Generating Cesium Dynamic Visualization

Now that we have the detailed "assignments" dataframe, we can generate a time-dynamic movie of the vehicle routes on a 3D map.

- See https://veroviz.org/docs/veroviz.createCesium.html for details on the function that makes this happen.
- See https://veroviz.org/documentation.html for instructions for viewing your Cesium movie.
- An example Cesium movie (for a different problem) is available here (no installation required):  https://veroviz.org/cesiumdemo.html

---

Before we continue, we need to know where Cesium was installed.
- One way to do specify Cesium's installation path is via system environment variables.  See https://veroviz.org/documentation.html for instructions.
- Alternatively, you can hard-code the path below.  On my Mac, I installed Cesium to `/Users/me/cesium`.  


In [39]:
# Specify the location where Cesium is installed on your computer:

# OPTION 1 -- Hard-coding the path here:
CESIUM_DIR = 'Users/me/cesium'

# OPTION 2 -- Using a system environment variable:
import os
CESIUM_DIR = os.environ['CESIUMDIR']

In [40]:
import os
vrv.createCesium(assignments = assignmentsDF, 
                 nodes       = myNodes, 
                 startDate   = '2019-10-20', 
                 startTime   = '08:00:00', 
                 postBuffer  = 30, 
                 cesiumDir   = CESIUM_DIR, 
                 problemDir  = 'INFORMS_demo')

Message: File selector was written to /home/cmurray3/cesium/INFORMS_demo/;INFORMS_demo.vrv ...
Message: Configs were written to /home/cmurray3/cesium/INFORMS_demo/config.js ...
Message: Nodes were written to /home/cmurray3/cesium/INFORMS_demo/displayNodes.js ...
Message: Assignments (.js) were written to /home/cmurray3/cesium/INFORMS_demo/displayPaths.js ...
Message: Assignments (.czml) were written to /home/cmurray3/cesium/INFORMS_demo/routes.czml ...


## Import/Export 

VeRoViz provides some utility functions to make it easy to save data you generated, and then import it later.  A few examples are shown below.

In [41]:
vrv.exportDataToCSV(timeSec, 'time.csv')
vrv.exportDataToCSV(distMeters, 'dist.csv')
vrv.exportDataToCSV(myNodes, 'nodes.csv')

# You may also wish to export your arcs and solutions.

Message: Data written to time.csv.
Message: Data written to dist.csv.
Message: Data written to nodes.csv.


In [42]:
importedNodes = vrv.importDataframe('nodes.csv')
importedNodes

,id,lat,lon,altMeters,nodeName,nodeType,popupText,leafletIconPrefix,leafletIconType,leafletColor,leafletIconText,cesiumIconType,cesiumColor,cesiumIconText,elevMeters
0,0,47.567258,-122.189889,0.0,Depot,depot,0,glyphicon,star,blue,0,pin,blue,0,NaN
1,1,47.608239,-122.324083,0.0,Cust 1,customer,1,glyphicon,home,red,1,pin,red,1,NaN
2,2,47.599200,-122.327818,0.0,Cust 2,customer,2,glyphicon,home,red,2,pin,red,2,NaN
3,3,47.627743,-122.311275,0.0,Cust 3,customer,3,glyphicon,home,red,3,pin,red,3,NaN
4,4,47.617816,-122.358286,0.0,Cust 4,customer,4,glyphicon,home,red,4,pin,red,4,NaN
5,5,47.621129,-122.362482,0.0,Cust 5,customer,5,glyphicon,home,red,5,pin,red,5,NaN
6,6,47.623110,-122.357715,0.0,Cust 6,customer,6,glyphicon,home,red,6,pin,red,6,NaN
7,7,47.617081,-122.357265,0.0,Cust 7,customer,7,glyphicon,home,red,7,pin,red,7,NaN


--- 

That brings us to the end of the material covered in the INFORMS 2019 talk.  There are loads of other features available in VeRoViz that just couldn't be covered in a 15-minute talk.  Here are some suggestions to help you learn more about VeRoViz:
- Experiment with the interactive "Sketch" web utility (https://veroviz.org/sketch.html).  You can create nodes and arcs via point-and-click, and export your creation into Python.
- Check out the installation guide:  https://veroviz.org/documentation.html 
- See some other Jupyter notebooks available in the Downloads section:  https://veroviz.org/downloads/ 
- Take a look at the extensive (perhaps overwhelming) documentation at https://veroviz.org/docs/.  Each VeRoViz function also includes examples.